# 📘 Quick Reference

**Purpose**: Process BraTS 2021 T1 MRI data for anomaly detection testing  
**Pipeline**: Extract → Smart Filter (Middle Slices) → Resize → ZIP  
**Output**: 128×128 normalized .npy files (same format as IXI training data)  
**Time**: ~60-70 minutes  

## 📋 Execution Order
1. **Cells 1-4**: Setup & explore (fast)
2. **Sections 1-8**: Setup, explore, define functions (fast)
3. **Section 9**: Extract ALL slices ⏰ 20-30 min → 97,882 slices
4. **Section 10**: Smart filter to middle 50% ⏰ 28 min → 49,888 slices  
5. **Section 11**: Resize to 128×128 ⏰ 30-40 min
6. **Section 15**: Create ZIP ⏰ 5-10 min

## ✅ Success = ~50,000 slices (3x IXI), 128×128, range [0,1], ZIP created

---

# BraTS 2021 T1 MRI Preprocessing Pipeline (Local → Drive Upload)

This notebook performs LOCAL preprocessing of BraTS 2021 T1 files with smart filtering:

## Pipeline Overview:
1. **Extract T1 files** from BraTS 2021 folders
2. **Extract 2D slices** from 3D T1 NIfTI volumes with validation
3. **Normalize** slices to [0, 1] range (per-slice normalization)
4. **Smart filter** to keep only middle 50% of slices (most informative)
5. **Resize** to 128x128 standard size
6. **Save as .npy files** ready for Google Drive upload

**Purpose**: BraTS data will be used for TESTING (anomaly detection), while IXI is for TRAINING  
**Result**: ~50,000 high-quality slices (3x IXI dataset size)

## 1. Import Required Libraries

In [ ]:
# Import required libraries
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from skimage.transform import resize
from datetime import datetime
import zipfile
from collections import defaultdict

print("✅ All libraries imported successfully")

In [ ]:
# Source folder with BraTS 2021 dataset
brats_folder = r"c:\Users\rifad\symAD-ECNN\data\brats2021"

# Output folders (clean structure under brats_t1/)
t1_raw_folder = r"c:\Users\rifad\symAD-ECNN\data\brats_t1\raw_slices"
filtered_folder = r"c:\Users\rifad\symAD-ECNN\data\brats_t1\filtered"
resized_folder = r"c:\Users\rifad\symAD-ECNN\data\brats_t1\resized"

# Create output directories
for folder in [t1_raw_folder, filtered_folder, resized_folder]:
    os.makedirs(folder, exist_ok=True)

print("📁 Folder Structure:")
print(f"  Source (BraTS 2021): {brats_folder}")
print(f"  Raw slices output: {t1_raw_folder}")
print(f"  Filtered output: {filtered_folder}")
print(f"  Resized output (128x128): {resized_folder}")
print(f"\n✓ Source folder exists: {os.path.exists(brats_folder)}")

📁 Folder Structure:
  Source (BraTS 2021): c:\Users\rifad\symAD-ECNN\data\brats2021
  Raw slices output: c:\Users\rifad\symAD-ECNN\data\brats2021_processed\raw_slices
  Filtered output: c:\Users\rifad\symAD-ECNN\data\brats2021_processed\filtered
  Resized output (128x128): c:\Users\rifad\symAD-ECNN\data\brats2021_processed\resized

✓ Source folder exists: True


## 2. Define Paths and Create Output Folders

In [ ]:
# Find all T1 files (not T1CE - contrast enhanced)
t1_files = []

for patient_folder in patient_folders:
    patient_path = os.path.join(brats_folder, patient_folder)
    # Look for files ending with _t1.nii.gz (not _t1ce.nii.gz)
    for file in os.listdir(patient_path):
        if file.endswith('_t1.nii.gz') and not file.endswith('_t1ce.nii.gz'):
            t1_files.append(os.path.join(patient_path, file))

print(f"Total T1 files found: {len(t1_files)}")
print(f"\nFirst 5 T1 files:")
for file in t1_files[:5]:
    print(f"  - {os.path.basename(file)}")

Total T1 files found: 1251

First 5 T1 files:
  - BraTS2021_00000_t1.nii.gz
  - BraTS2021_00002_t1.nii.gz
  - BraTS2021_00003_t1.nii.gz
  - BraTS2021_00005_t1.nii.gz
  - BraTS2021_00006_t1.nii.gz


## 3. Explore Dataset Structure

In [ ]:
# Explore the dataset structure
patient_folders = sorted([f for f in os.listdir(brats_folder) if os.path.isdir(os.path.join(brats_folder, f))])

print(f"📊 Dataset Summary:")
print(f"  Total patient folders: {len(patient_folders)}")
print(f"\nFirst 5 patients:")
for folder in patient_folders[:5]:
    print(f"  - {folder}")
    
# Check what files are in first patient folder
if patient_folders:
    first_patient = os.path.join(brats_folder, patient_folders[0])
    files = os.listdir(first_patient)
    print(f"\nFiles in {patient_folders[0]}:")
    for f in files:
        print(f"  - {f}")

## 4. Find All T1 Files

## 5. STEP 1: Extract ALL Slices with RAS Orientation ⏰ 20-30 min

**CRITICAL**: Apply RAS orientation correction to match IXI preprocessing

Extract 2D slices from all BraTS T1 volumes with proper orientation standardization.

**Output**: ~97,882 raw slices (all slices from all patients with RAS orientation)

In [ ]:
# SECTION 9: Extract ALL slices from 3D volumes with RAS orientation correction
import os
import numpy as np
import nibabel as nib
from tqdm import tqdm

# Preprocessing functions
def normalize(x):
    """Normalize array to [0, 1] range"""
    x = x.astype(np.float32)
    if x.max() - x.min() < 1e-6:
        return np.zeros_like(x)
    return (x - x.min()) / (x.max() - x.min())

def is_valid_slice(slice_array, min_nonzero_ratio=0.12, min_mean=0.1):
    """Check if slice contains meaningful information"""
    nonzero_ratio = np.count_nonzero(slice_array) / slice_array.size
    if nonzero_ratio < min_nonzero_ratio:
        return False
    
    normalized = normalize(slice_array)
    if normalized.mean() < min_mean:
        return False
    
    return True

# Process all T1 files
print("="*70)
print("EXTRACTING SLICES WITH RAS ORIENTATION CORRECTION")
print("="*70)
print(f"\nProcessing {len(t1_files)} T1 files...")
print(f"Output: {t1_raw_folder}")

slice_idx = 0
skipped_count = 0

for t1_file in tqdm(t1_files, desc="Processing T1 volumes"):
    try:
        # ✅ CRITICAL: Apply RAS orientation correction
        nii_img = nib.load(t1_file)
        nii_img = nib.as_closest_canonical(nii_img)  # Force RAS orientation
        vol = nii_img.get_fdata()
        
        # Extract 2D slices (axial view - axis 2)
        Z = vol.shape[2]
        for s in range(Z):
            slice_2d = vol[:, :, s]
            
            # Skip empty or low-information slices
            if not is_valid_slice(slice_2d, min_nonzero_ratio=0.12, min_mean=0.1):
                skipped_count += 1
                continue
            
            # Normalize to [0, 1]
            normalized_slice = normalize(slice_2d)
            
            # Save as .npy file
            output_path = os.path.join(t1_raw_folder, f"slice_{slice_idx:06d}.npy")
            np.save(output_path, normalized_slice)
            slice_idx += 1
            
    except Exception as e:
        print(f"\n❌ Error processing {os.path.basename(t1_file)}: {e}")
        continue

print(f"\n" + "="*70)
print("EXTRACTION COMPLETE")
print("="*70)
print(f"✅ Total slices extracted: {slice_idx:,}")
print(f"✅ Skipped (empty/low-info): {skipped_count:,}")
print(f"✅ Output location: {t1_raw_folder}")
print("="*70)

## 6. STEP 2: Smart Filter - Keep Middle 50% Slices ⏰ 28 min

Filter to keep only the most informative slices (middle 50% from each patient).

**Why**: Middle slices contain brain regions where tumors typically appear  
**Output**: ~49,888 filtered slices

In [ ]:
# SECTION 10: Smart filtering - keep middle 50% of slices per patient
from collections import defaultdict

print("="*70)
print("SMART FILTERING - MIDDLE 50% SLICES")
print("="*70)

# Get all raw files
raw_files = sorted([f for f in os.listdir(t1_raw_folder) if f.endswith('.npy')])
print(f"\nTotal raw slices: {len(raw_files):,}")

# Calculate slices per patient (approximate)
total_slices = len(raw_files)
num_patients = len(t1_files)
slices_per_patient = total_slices // num_patients

print(f"Patients: {num_patients}")
print(f"Avg slices per patient: {slices_per_patient}")

# Keep middle 50% from each patient group
MIDDLE_PERCENTAGE = 0.5
start_margin = int(slices_per_patient * (1 - MIDDLE_PERCENTAGE) / 2)
end_margin = slices_per_patient - start_margin

print(f"\nFiltering strategy:")
print(f"  - Keep middle {MIDDLE_PERCENTAGE*100:.0f}% per patient")
print(f"  - Skip first {start_margin} and last {start_margin} slices per patient")

# Determine which slices to keep
slices_to_keep = set()
for patient_idx in range(num_patients):
    patient_start = patient_idx * slices_per_patient
    keep_start = patient_start + start_margin
    keep_end = patient_start + end_margin
    
    for slice_idx in range(keep_start, keep_end):
        if slice_idx < total_slices:
            slices_to_keep.add(slice_idx)

print(f"\nProcessing {len(slices_to_keep):,} slices...")

# Copy filtered slices
kept_count = 0
for idx, filename in enumerate(tqdm(raw_files, desc="Filtering")):
    if idx in slices_to_keep:
        src = os.path.join(t1_raw_folder, filename)
        dst = os.path.join(filtered_folder, filename)
        
        arr = np.load(src)
        np.save(dst, arr)
        kept_count += 1

print(f"\n" + "="*70)
print("FILTERING COMPLETE")
print("="*70)
print(f"✅ Slices kept: {kept_count:,} / {total_slices:,} ({kept_count/total_slices*100:.1f}%)")
print(f"✅ Output location: {filtered_folder}")
print("="*70)

## 7. STEP 3: Resize to 128×128 with Bicubic Interpolation ⏰ 30-40 min

**CRITICAL**: Use bicubic interpolation (order=3) for SHARP images (not blurry)

Resize all filtered slices to 128×128 to match the IXI dataset dimensions.

**Input**: filtered/ folder with ~49,888 middle slices  
**Output**: resized/ folder with 128×128 .npy files

In [ ]:
# SECTION 11: Resize to 128×128 with BICUBIC interpolation (sharper than linear)
from skimage.transform import resize

print("="*70)
print("RESIZING TO 128×128 WITH BICUBIC INTERPOLATION")
print("="*70)

# Get filtered files
filtered_files = sorted([f for f in os.listdir(filtered_folder) if f.endswith('.npy')])
total = len(filtered_files)

print(f"\nTotal files to resize: {total:,}")
print(f"Output: {resized_folder}")

# Batch configuration
BATCH_SIZE = 500
TARGET_SIZE = (128, 128)

resized_count = 0
error_count = 0

for i in range(0, total, BATCH_SIZE):
    batch_files = filtered_files[i : i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    
    print(f"\nProcessing batch {batch_num} ({len(batch_files)} files)...")
    
    for f in tqdm(batch_files, desc=f"Batch {batch_num}"):
        src_path = os.path.join(filtered_folder, f)
        dst_path = os.path.join(resized_folder, f)
        
        # Skip if already resized
        if os.path.exists(dst_path):
            resized_count += 1
            continue
        
        try:
            arr = np.load(src_path)
            
            # ✅ CRITICAL: Use BICUBIC interpolation (order=3) for sharp images
            # NOTE: order=3 prevents blurriness caused by order=1 (linear)
            resized = resize(
                arr,
                TARGET_SIZE,
                order=3,              # Bicubic (sharper than default linear order=1)
                mode='reflect',       # Reflect padding at edges
                anti_aliasing=True,   # Prevent jagged artifacts
                preserve_range=True   # Maintain [0,1] normalization
            )
            
            np.save(dst_path, resized.astype(np.float32))
            resized_count += 1
            
        except Exception as e:
            print(f"❌ Error resizing {f}: {e}")
            error_count += 1
            continue

print(f"\n" + "="*70)
print("RESIZING COMPLETE")
print("="*70)
print(f"✅ Successfully resized: {resized_count:,}")
print(f"✅ Errors: {error_count}")
print(f"✅ Output location: {resized_folder}")
print("="*70)

In [ ]:
# Verify the final resized files
resized_files = sorted([f for f in os.listdir(resized_folder) if f.endswith('.npy')])

print(f"📊 Final processed files: {len(resized_files)}")

# Load and inspect a sample file
if resized_files:
    sample_file = resized_files[0]
    sample = np.load(os.path.join(resized_folder, sample_file))

    print(f"\n✓ Sample file: {sample_file}")
    print(f"  - Shape: {sample.shape}")
    print(f"  - Min pixel value: {sample.min():.4f}")
    print(f"  - Max pixel value: {sample.max():.4f}")
    print(f"  - Mean pixel value: {sample.mean():.4f}")
    print(f"  - Data type: {sample.dtype}")

    # Verify dimensions match IXI
    if sample.shape == (128, 128):
        print(f"\n✅ Dimensions match IXI dataset (128x128)")
    else:
        print(f"\n⚠️ Warning: Dimensions don't match expected (128x128)")

📊 Final processed files: 49888

✓ Sample file: slice_000038.npy
  - Shape: (128, 128)
  - Min pixel value: 0.0000
  - Max pixel value: 0.8583
  - Mean pixel value: 0.1507
  - Data type: float32

✅ Dimensions match IXI dataset (128x128)


## 8. Verify Resized Output

## 9. Create Filtered Subset for Fast Evaluation ⏰ 5 min

**Purpose**: Reduce dataset size from 49,888 → ~5,000 slices for faster model evaluation

**Rationale**:
- **Development Speed**: Full dataset evaluation takes ~40 minutes, filtered takes ~2 minutes (95% faster)
- **Research Validity**: Stratified sampling maintains statistical integrity:
  - All 1,251 patients represented (no selection bias)
  - Middle slices contain most informative brain regions (where tumors typically appear)
  - Standard practice in medical imaging research (papers commonly use 5-20 slices per volume)
- **Sampling Strategy**:
  - Take 4 slices per patient from middle 60% range (skip empty edge slices)
  - Evenly spaced to capture structural variation
- **Full Dataset Available**: Original 49,888 slices preserved for final validation if needed

In [ ]:
# SECTION 15: Create filtered subset (4 slices per patient for fast evaluation)
from datetime import datetime
import zipfile

print("="*70)
print("CREATING FILTERED SUBSET FOR EVALUATION")
print("="*70)

# Create eval folder
filtered_eval_folder = os.path.join(os.path.dirname(resized_folder), "filtered_eval")
os.makedirs(filtered_eval_folder, exist_ok=True)

# Stratified sampling: 4 slices per patient from middle 60%
TARGET_PER_PATIENT = 4
MIDDLE_RANGE = 0.6

resized_files = sorted([f for f in os.listdir(resized_folder) if f.endswith('.npy')])
total_slices = len(resized_files)
num_patients = len(t1_files)
slices_per_patient = total_slices // num_patients

print(f"\nTotal resized slices: {total_slices:,}")
print(f"Patients: {num_patients}")
print(f"Target per patient: {TARGET_PER_PATIENT}")

# Sample evenly from middle range
filtered_count = 0
margin = int(slices_per_patient * (1 - MIDDLE_RANGE) / 2)
step = (slices_per_patient - 2 * margin) // TARGET_PER_PATIENT

for patient_idx in tqdm(range(num_patients), desc="Sampling patients"):
    patient_start = patient_idx * slices_per_patient
    sample_start = patient_start + margin
    
    for i in range(TARGET_PER_PATIENT):
        slice_idx = sample_start + i * step
        if slice_idx < len(resized_files):
            src_file = resized_files[slice_idx]
            src_path = os.path.join(resized_folder, src_file)
            dst_path = os.path.join(filtered_eval_folder, src_file)
            
            if os.path.exists(src_path):
                arr = np.load(src_path)
                np.save(dst_path, arr)
                filtered_count += 1

print(f"\n✅ Filtered subset created: {filtered_count:,} slices")
print(f"✅ Output location: {filtered_eval_folder}")

# Create ZIP file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"brats2021_filtered_RAS_4slices_{timestamp}.zip"
zip_path = os.path.join(os.path.dirname(resized_folder), zip_filename)

print(f"\n📦 Creating ZIP file: {zip_filename}")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    eval_files = sorted([f for f in os.listdir(filtered_eval_folder) if f.endswith('.npy')])
    for filename in tqdm(eval_files, desc="Compressing"):
        file_path = os.path.join(filtered_eval_folder, filename)
        zipf.write(file_path, filename)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print(f"\n" + "="*70)
print("SUBSET & ZIP CREATION COMPLETE")
print("="*70)
print(f"✅ Evaluation slices: {filtered_count:,}")
print(f"✅ ZIP file: {zip_filename}")
print(f"✅ ZIP size: {zip_size_mb:.1f} MB")
print(f"✅ Location: {zip_path}")
print(f"\n📤 Upload this ZIP to Google Drive:")
print(f"   MyDrive/symAD-ECNN/data/")
print("="*70)

## 10. Upload Instructions

**Next Steps**:
1. Upload the ZIP file created above to Google Drive
2. Path: `MyDrive/symAD-ECNN/data/`
3. Then run the extraction code below in Google Colab

## 11. Colab Extraction Code

**Run this in Google Colab** after uploading the ZIP file:

In [ ]:
# ============================================================
# BraTS ZIP Extraction in Google Colab
# ============================================================
# Copy this code and run in Colab after uploading the ZIP

from google.colab import drive
import zipfile
import os

# Mount Drive
drive.mount('/content/drive')

# Paths
BASE = "/content/drive/MyDrive/symAD-ECNN/data"
ZIP_FILE = f"{BASE}/brats2021_filtered_RAS_4slices_YYYYMMDD_HHMMSS.zip"  # Update timestamp
OUTPUT_FOLDER = f"{BASE}/brats_t1/test"

# Create output folder
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Extract ZIP
print(f"Extracting {os.path.basename(ZIP_FILE)}...")
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(OUTPUT_FOLDER)

# Verify extraction
extracted_files = [f for f in os.listdir(OUTPUT_FOLDER) if f.endswith('.npy')]
print(f"\n✓ Extracted {len(extracted_files):,} files to {OUTPUT_FOLDER}")
print(f"✓ Ready for CNN-AE testing!")